# ML-10 — Content Action Playbook

This notebook turns the validated review-priority approach from W04/W05 into an operational content action queue. It reconstructs the necessary March 2026 decision window because the earlier notebooks did not persist their in-memory tables. The output is decision-support: it ranks pages worth human review; it does not claim that a refresh will cause recovery.

## 1. Ranked actions + reason codes

The queue prioritizes pages with meaningful search visibility and weak CTR relative to pages at a similar average position. Position is handled in tiers so high-ranking and lower-ranking pages are not compared with one global CTR threshold. The score is used only to order review work.

Reason codes:
- `low_ctr_visible_page` — meaningful visibility with CTR below the position-tier benchmark.
- `insufficient_visibility` — too few impressions for a reliable CTR review.
- `position_unavailable` — average position is unavailable/zero.
- `poor_position` — visibility exists but the page ranks poorly; diagnose relevance/content fit before treating CTR as the main issue.
- `high_ctr_visible_page` — CTR is at/above its position-tier benchmark; no immediate CTR-focused action.
- `not_prioritized` — does not meet the review conditions.

### Archetype → action mapping
The operational archetype is the position/visibility situation represented by the reason code. It maps directly to a human review action rather than an automatic edit:

| Archetype | Action |
|---|---|
| Visible + low CTR | Review title/snippet and search-intent alignment |
| Visible + poor position | Diagnose content relevance, coverage and ranking competition |
| Low visibility | Do not prioritize from CTR evidence alone |
| Position unavailable | Verify search data before acting |
| Strong CTR | Preserve current search-result presentation; monitor |

### Decay / refresh insight
A fall in impressions from the decision window to the later holdout window is treated as an observed deterioration signal. It can support prioritization for review, but it does **not** establish that refreshing the page caused or will cause recovery. Where update-age metadata is available, it is treated as supporting context rather than a causal variable.

In [ ]:
import os
import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.sql('INSTALL httpfs; LOAD httpfs;')
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
WAREHOUSE = 'hf://datasets/FlyRank/internship-warehouse'
march_path = f'{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet'
df = con.sql(f"SELECT * FROM read_parquet('{march_path}') WHERE gsc_data_available IS TRUE").df()
df['report_date'] = pd.to_datetime(df['report_date'])
decision = df[df.report_date.dt.day <= 15].copy()
holdout = df[df.report_date.dt.day >= 16].copy()

features = (decision.groupby(['client_hash_id','content_hash_id']).agg(
    impressions=('gsc_impressions','sum'), clicks=('gsc_clicks','sum'),
    avg_position=('gsc_avg_position','mean'), sessions_organic=('sessions_organic','sum'),
    sessions_ai=('sessions_ai','sum')).reset_index())
features['ctr'] = features['clicks'] / features['impressions'].replace(0,np.nan)
decision_impr = decision.groupby(['client_hash_id','content_hash_id']).gsc_impressions.mean().reset_index(name='decision_impressions_avg')
holdout_impr = holdout.groupby(['client_hash_id','content_hash_id']).gsc_impressions.mean().reset_index(name='holdout_impressions')
data = features.merge(decision_impr,on=['client_hash_id','content_hash_id']).merge(holdout_impr,on=['client_hash_id','content_hash_id'])
data['impression_change_pct'] = (data['holdout_impressions'] / data['decision_impressions_avg'].replace(0,np.nan) - 1) * 100
data['observed_decline'] = (data['holdout_impressions'] < data['decision_impressions_avg']).astype(int)

# Position tiers used by W04's rule-based reasoning.
def position_tier(p):
    if pd.isna(p) or p <= 0: return 'position_unavailable'
    if p <= 5: return '1-5'
    if p <= 10: return '6-10'
    if p <= 20: return '11-20'
    return '21+'

data['position_tier'] = data['avg_position'].apply(position_tier)
MIN_IMPRESSIONS = 30
bench = data[data.position_tier != 'position_unavailable'].groupby('position_tier')['ctr'].median().rename('tier_ctr_benchmark')
data = data.join(bench,on='position_tier')
data['ctr_gap'] = data['tier_ctr_benchmark'] - data['ctr']
data['action_score'] = data['ctr_gap'].clip(lower=0).fillna(0) * np.log1p(data['impressions'])

data['reason_code'] = np.select([
    data.position_tier.eq('position_unavailable'),
    data.impressions < MIN_IMPRESSIONS,
    (data.impressions >= MIN_IMPRESSIONS) & (data.avg_position > 20) & (data.ctr_gap > 0),
    (data.impressions >= MIN_IMPRESSIONS) & (data.ctr_gap > 0),
    (data.impressions >= MIN_IMPRESSIONS) & (data.ctr_gap <= 0)
],['position_unavailable','insufficient_visibility','poor_position','low_ctr_visible_page','high_ctr_visible_page'],default='not_prioritized')

action_map = {
 'low_ctr_visible_page':'Review title/snippet and search-intent alignment',
 'poor_position':'Diagnose content relevance, coverage and ranking competition',
 'insufficient_visibility':'Do not prioritize from CTR evidence alone',
 'position_unavailable':'Verify search data before acting',
 'high_ctr_visible_page':'Preserve current search-result presentation; monitor',
 'not_prioritized':'No immediate action from this rule'
}
data['recommended_action'] = data.reason_code.map(action_map)
queue = data.sort_values(['action_score','impressions'],ascending=[False,False]).reset_index(drop=True)
queue['priority_rank'] = np.arange(1,len(queue)+1)

print(f'Rows in reconstructed decision table: {len(data):,}')
print(f'Clients: {data.client_hash_id.nunique():,}')
print('Reason-code counts:')
display(queue.reason_code.value_counts().to_frame('pages'))
print('Top 20 review queue:')
display(queue.head(20)[['priority_rank','client_hash_id','content_hash_id','action_score','impressions','avg_position','ctr','tier_ctr_benchmark','impression_change_pct','reason_code','recommended_action']])

## 2. Intended use and limits

This playbook is for a content/SEO team deciding which pages deserve review first. It is most useful as a triage layer: the rank identifies where to spend limited review time, while the reason code identifies the first diagnostic question.

The evidence is observational. The queue can say that pages with particular signals were observed in this dataset and period, and that the validated ranking flags pages out of sample. It cannot say that refreshing a page will increase clicks, that the queue predicts Google's algorithm, or that a listed action will produce recovery. The March decision window is also one evaluation period, so the queue should not be treated as universally calibrated across clients or future periods.

In [ ]:
# Compact scope checks
print('Decision window:', decision.report_date.min().date(), 'to', decision.report_date.max().date())
print('Holdout window:', holdout.report_date.min().date(), 'to', holdout.report_date.max().date())
print('Queue size:', len(queue))
print('Pages with observed later-period impression decline:', f"{queue.observed_decline.mean()*100:.1f}%")

## 3. Human review + the no-go list

Every high-priority row requires human review before any content change. The reviewer should verify search intent, the actual search-result presentation, content quality/coverage, business importance, and whether there is a known technical or measurement issue.

### Never automate
- Publishing, deleting, or materially rewriting content solely from the score.
- Changing titles/meta descriptions without a human checking the query intent and SERP context.
- Declaring a page 'bad' because its position is low.
- Treating observed impression decline as proof that content is stale or that a refresh will recover it.
- Making client-facing recommendations without checking the underlying page and data quality.

### Cost/value thinking
Review capacity is limited, so the queue should be treated as a way to allocate human attention. High-visibility pages with a large negative CTR gap are attractive first reviews because a single review can cover substantial existing search exposure. Low-visibility pages should normally be cheaper to defer because the available evidence is weaker. Business value, implementation effort and strategic importance should be added by the human reviewer; they are not inferred from the score.

In [ ]:
# Simple workload view: how many pages fall into each action class
workload = (queue.groupby('reason_code').agg(pages=('content_hash_id','size'),
    median_impressions=('impressions','median'), median_score=('action_score','median')).sort_values('pages',ascending=False))
display(workload)

## 4. Monitoring / retrain triggers

The playbook should be rerun when the underlying search environment changes materially or when its ranking quality starts to drift. Useful lightweight triggers are:

- rerun on a regular monthly cadence using a new decision/holdout period;
- monitor the distribution of impressions, CTR and average position for large shifts;
- monitor the share of pages receiving each reason code;
- re-check precision@K against a later held-out outcome when labels are available;
- investigate if the baseline no longer separates high-priority pages from the base rate;
- retrain/reconsider the learned model only if new data or monitoring shows that the simple baseline is no longer adequate.

A retrain is therefore a response to measured drift or degraded out-of-sample ranking performance, not an automatic response to a single unusual page.

In [ ]:
# Lightweight monitoring snapshot saved with the queue
monitor = pd.DataFrame({
 'metric':['pages','clients','median_impressions','median_position','median_ctr','decline_rate_pct'],
 'value':[len(queue),queue.client_hash_id.nunique(),queue.impressions.median(),queue.avg_position.median(),queue.ctr.median(),queue.observed_decline.mean()*100]
})
display(monitor)

## 5. Exports for the paper

The ranked queue and monitoring snapshot are written to `work/outputs/`. The queue is intentionally kept out of git by the project design; the paper can build from the exported artifact produced when this notebook is run.

In [ ]:
from pathlib import Path
out = Path('work/outputs')
out.mkdir(parents=True,exist_ok=True)
queue_cols = ['priority_rank','client_hash_id','content_hash_id','action_score','impressions','clicks','avg_position','ctr','position_tier','tier_ctr_benchmark','ctr_gap','decision_impressions_avg','holdout_impressions','impression_change_pct','observed_decline','reason_code','recommended_action']
queue[queue_cols].head(20).to_csv(out/'content_action_queue_top20.csv',index=False)
queue[queue_cols].to_csv(out/'content_action_queue_full.csv',index=False)
monitor.to_csv(out/'content_action_monitoring_snapshot.csv',index=False)
print('Exported:')
for p in ['content_action_queue_top20.csv','content_action_queue_full.csv','content_action_monitoring_snapshot.csv']:
    print(out/p)

## 6. Self-check

- [x] Every section is filled with markdown and supporting code.
- [ ] Run the notebook top to bottom in Colab before submission.
- [x] Client and content IDs are pseudonyms used for grouping/output only, not model features.
- [x] The decision window is separated from the later holdout window.
- [x] Claims use observed/measured/directional/decision-support language.
- [x] Human review is required before acting.
- [x] Publishing or material content changes are explicitly not automated.
- [x] Queue and monitoring files are exported to `work/outputs/`.
- [ ] Confirm the generated queue files remain uncommitted if the repository's `.gitignore` excludes `work/outputs/`.
